# Unlimited-OCR Colab server (POC)

Hosts `baidu/Unlimited-OCR` behind FastAPI, exposed via a Cloudflare quick tunnel (no account or signup required).

**Setup before running:**
1. Runtime → Change runtime type → **GPU** (T4 is fine).
2. Colab Secrets: add optional `UOCR_API_KEY` (leave unset to disable auth).
3. Run all cells top-to-bottom. Copy the printed `PUBLIC URL` (`https://….trycloudflare.com`) into local `.env` as `UNLIMITED_OCR_REMOTE_URL`.

VRAM ≥ 80% → whole model offloads to CPU for that request.

In [ ]:
# 1) Install deps (Colab already has torch + CUDA)
!pip -q install fastapi "uvicorn[standard]" nest_asyncio python-multipart \
  "transformers>=4.57.0,<4.58.0" einops addict easydict

In [ ]:
# 2) Secrets — set in Colab UI: Secrets → UOCR_API_KEY (optional shared secret)
from google.colab import userdata

API_KEY = userdata.get("UOCR_API_KEY")  # leave unset to disable auth
print("API_KEY auth:", "enabled" if API_KEY else "disabled (open endpoint)")

In [ ]:
# 3) Load model once onto CUDA
import torch
from transformers import AutoModel, AutoTokenizer

assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type → GPU)"

MODEL_NAME = "baidu/Unlimited-OCR"
print(f"Loading {MODEL_NAME} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
).eval().cuda()
CURRENT_DEVICE = "cuda"
print(f"Model ready on {CURRENT_DEVICE}")

In [ ]:
# 4) VRAM monitor + offload gate (threshold 80%)
import logging
import threading
import time

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
log = logging.getLogger("uocr")

_lock = threading.Lock()
VRAM_THRESHOLD = 0.80


def vram_usage_ratio() -> float:
    free, total = torch.cuda.mem_get_info(0)
    return 1.0 - (free / total)


def run_inference(image_path: str, out_dir: str) -> dict:
    """Run model.infer; offload whole model to CPU if VRAM usage >= 80%."""
    global model, CURRENT_DEVICE
    with _lock:
        ratio = vram_usage_ratio()
        target = "cpu" if ratio >= VRAM_THRESHOLD else "cuda"
        if target != CURRENT_DEVICE:
            log.info("VRAM %.0f%% -> switching model to %s", ratio * 100, target)
            model = model.to(target)
            CURRENT_DEVICE = target
            if target == "cuda":
                torch.cuda.empty_cache()
        t0 = time.time()
        model.infer(
            tokenizer,
            prompt="<image>document parsing.",
            image_file=image_path,
            output_path=out_dir,
            base_size=1024,
            image_size=640,
            crop_mode=True,
            max_length=32768,
            no_repeat_ngram_size=35,
            ngram_window=128,
            save_results=True,
        )
        elapsed = time.time() - t0
        log.info(
            "infer done device=%s elapsed=%.2fs vram=%.0f%%",
            CURRENT_DEVICE,
            elapsed,
            ratio * 100,
        )
        return {
            "device": CURRENT_DEVICE,
            "vram_pct": round(ratio * 100),
            "elapsed_sec": round(elapsed, 2),
        }


def _vram_watchdog():
    while True:
        try:
            log.info("VRAM watchdog: %.0f%% (device=%s)", vram_usage_ratio() * 100, CURRENT_DEVICE)
        except Exception as exc:
            log.warning("VRAM watchdog error: %s", exc)
        time.sleep(30)


threading.Thread(target=_vram_watchdog, daemon=True).start()
print(f"VRAM threshold={VRAM_THRESHOLD:.0%}; watchdog every 30s")

In [ ]:
# 5) FastAPI app
import pathlib
import shutil
import tempfile

from fastapi import Depends, FastAPI, File, Header, HTTPException, UploadFile

app = FastAPI(title="Unlimited-OCR Colab")


def _check_key(x_api_key: str | None = Header(default=None)):
    if API_KEY and x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="bad api key")


@app.get("/health")
def health(_=Depends(_check_key)):
    return {
        "status": "ok",
        "model": MODEL_NAME,
        "device": CURRENT_DEVICE,
        "vram_pct": round(vram_usage_ratio() * 100),
    }


@app.post("/infer")
async def infer(file: UploadFile = File(...), _=Depends(_check_key)):
    tmp = tempfile.mkdtemp(prefix="uocr_")
    try:
        img_path = pathlib.Path(tmp) / "page.jpg"
        img_path.write_bytes(await file.read())
        meta = run_inference(str(img_path), tmp)
        md_path = pathlib.Path(tmp) / "result.md"
        markdown = md_path.read_text(encoding="utf-8").strip() if md_path.exists() else ""
        return {"markdown": markdown, **meta}
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


print("FastAPI routes: GET /health, POST /infer")

In [ ]:
# 6) Download cloudflared, start uvicorn in the background, start a quick tunnel (blocking — keeps Colab session alive)
import re
import subprocess
import threading
import time

import nest_asyncio
import uvicorn

nest_asyncio.apply()

!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared


def _run_api():
    uvicorn.run(app, host="0.0.0.0", port=8001)


threading.Thread(target=_run_api, daemon=True).start()
time.sleep(2)  # let uvicorn bind before the tunnel connects

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
print("Starting Cloudflare quick tunnel …")
for line in proc.stdout:
    print(line, end="")
    if public_url is None:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print("\nPUBLIC URL:", public_url)
            print("Set UNLIMITED_OCR_REMOTE_URL to that URL in local .env, then restart uvicorn.")
